# Table-tap transient analysis

This notebook explores transient vibration caused by table taps, including centering, RMS, peak-to-peak range, and clipping checks. It helps validate behavior outside the controlled training conditions.

> Keep reusable calculations in `src/machine_sentinel/features.py`; this notebook is for interactive inspection and plots.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('../../data/raw/table_taps.csv')

In [ ]:
df.head(5)

In [ ]:
df['time_s'] = (df['timestamp_us'] - df['timestamp_us'].iloc[0]) / 1e6

In [ ]:
accl_z = df['az']
time = df['time_s']

In [ ]:
mean = np.mean(accl_z)
std = np.std(accl_z)
ptp = np.ptp(accl_z)
rms = np.sqrt(np.mean(np.square(accl_z)))

In [ ]:
print(f"Mean: {mean}")
print(f"Standard Deviation: {std}")
print(f"Peak-to-Peak: {ptp}")
print(f"RMS: {rms}")

In [ ]:
z_centered = accl_z - mean

In [ ]:
plt.plot(time, z_centered, label='Z-Centered Acceleration')
plt.xlabel('Time (s)')
plt.ylabel('Z-Centered Acceleration')
plt.title('Table Taps Analysis')
plt.legend()
plt.show()

In [ ]:
rms_z_centered = np.sqrt(np.mean(np.square(z_centered)))
print(f"RMS of Z-Centered Acceleration: {rms_z_centered}")

mean_z_centered = np.mean(z_centered)
print(f"Mean of Z-Centered Acceleration: {mean_z_centered}")

std_z_centered = np.std(z_centered)
print(f"Standard Deviation of Z-Centered Acceleration: {std_z_centered}")

ptp_z_centered = np.ptp(z_centered)
print(f"Peak-to-Peak of Z-Centered Acceleration: {ptp_z_centered}")

In [ ]:
print("Minimum:", df["az"].min())
print("Maximum:", df["az"].max())

print("At minimum:",
      (df["az"] == -32768).sum())

print("At maximum:",
      (df["az"] == 32767).sum())

In [ ]:
clipped = (df["az"] == 32767) | (df["az"] == -32768)

print("Clipped samples:", clipped.sum())
print("Clipped percentage:", clipped.mean() * 100)

In [ ]:
import pandas as pd
import numpy as np

SAMPLE_RATE = 200
WINDOW_SIZE = 200       # 1 second
COUNTS_PER_G = 4096.0   # MPU6050 at ±8g


def extract_features(df, experiment, label=None):

    features = []

    for start in range(0, len(df), WINDOW_SIZE):

        end = start + WINDOW_SIZE

        # Ignore incomplete final window
        if end > len(df):
            break

        window = df.iloc[start:end]

        # Convert raw Z-axis counts -> g
        z_g = window["az"] / COUNTS_PER_G

        # Remove gravity / DC component
        z_centered = z_g - z_g.mean()

        # Time-domain features
        rms = np.sqrt(np.mean(np.square(z_centered)))

        # ddof=0 so STD matches centered RMS mathematically
        std = z_centered.std(ddof=0)

        peak_to_peak = z_g.max() - z_g.min()

        features.append({
            "start_idx": start,
            "end_idx": end,
            "start_time_s": start / SAMPLE_RATE,

            "rms": rms,
            "std": std,
            "ptp": peak_to_peak,

            "experiment": experiment,
            "label": label
        })

    return pd.DataFrame(features)

In [ ]:
tap_features = extract_features(
    df,
    experiment="tap_test",
    label=None
)

print(tap_features)